# 중간고사 대체 실습과제 — 문제 3

## 구조적 사고: MECE와 로직 트리

| 항목 | 내용 |
|------|------|
| **관련 챕터** | Ch03. 구조적 사고: MECE와 로직 트리 |
| **핵심 개념** | MECE 원칙, Why Tree, How Tree |
| **배점** | 25점 |
| **제출** | 이 노트북(.ipynb)을 실행 결과 포함하여 제출 |

---

## 시나리오

**프레시밀** CEO가 긴급 회의를 소집했습니다.

> *"매출이 하락하고 있습니다. 어디서 문제가 생겼는지 구조적으로 분석하고,
> 고객 이탈 원인을 정리해주세요."*

매출 구조 데이터와 이탈 사유 데이터가 제공됩니다.

In [1]:
# 환경설정
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# 데이터 로드
revenue = pd.read_csv('data/revenue_breakdown.csv')
churn = pd.read_csv('data/churn_reasons.csv')

print('=== 매출 구조 ===')
print(revenue.to_string(index=False))
print(f'\n총 매출: {revenue["Value_억원"].sum()}억 원')
print(f'\n=== 이탈 사유 (3개월 합계) ===')
print(churn.groupby('Reason_Category')['Churn_Count'].sum().sort_values(ascending=False).to_string())

=== 매출 구조 ===
Level1 Level2 Level3  Value_억원  Growth_pct
   온라인    구독형   정기배송        45         -12
   온라인    구독형   맞춤식단        28           5
   온라인   단건주문    자사앱        35          -8
   온라인   단건주문  배달앱입점        22         -25
  오프라인    자판기    사무실         8          15
  오프라인    자판기    헬스장         5          20
  오프라인  팝업스토어    백화점        12          -5
  오프라인  팝업스토어    대학교         6          10

총 매출: 161억 원

=== 이탈 사유 (3개월 합계) ===
Reason_Category
서비스품질    77
제품       30
가격       19
경쟁사      12


---

## 과제 1. MECE 검증과 매출 분해 (9점)

### 요구사항

1. **(3점)** 매출 데이터의 분류(Level1 → Level2 → Level3)가 **MECE를 만족하는지** 판단하세요.
   - 상호 배타적(ME)인가? 전체 포괄적(CE)인가?
   - 빠진 항목이 있다면 어떤 것인지 서술

2. **(3점)** 매출 데이터를 **Sunburst 차트** 또는 **Treemap**으로 시각화하세요.
   - Level1 → Level2 → Level3 계층 구조가 보여야 합니다

3. **(3점)** 성장률이 마이너스인 항목만 추출하여, 이 항목들의 매출이 전체에서 차지하는 **비율(%)**을 계산하고 `print()`로 출력하세요.

In [3]:
# ✏️ 과제 1-1: Sunburst 또는 Treemap 시각화
import plotly.express as px

# 데이터 준비 (열 이름 변경)
df = revenue.copy()
df.rename(columns={'Value_억원': 'Revenue', 'Growth_pct': 'Growth'}, inplace=True)

print("Treemap 시각화 데이터:")
print(df.to_string(index=False))
print()

# Treemap 시각화
fig = px.treemap(
    df,
    path=['Level1', 'Level2', 'Level3'],
    values='Revenue',
    color='Growth',
    color_continuous_scale='RdYlGn',
    hover_data={'Revenue': ':.0f', 'Growth': ':.1f'}
)

fig.update_layout(
    title='매출 구조 Treemap (성장률 색상 표시)',
    font=dict(color='black'),
    height=700,
    coloraxis_colorbar=dict(
        title='성장률(%)',
        tickfont=dict(color='black')
    )
)

fig.show()

Treemap 시각화 데이터:
Level1 Level2 Level3  Revenue  Growth
   온라인    구독형   정기배송       45     -12
   온라인    구독형   맞춤식단       28       5
   온라인   단건주문    자사앱       35      -8
   온라인   단건주문  배달앱입점       22     -25
  오프라인    자판기    사무실        8      15
  오프라인    자판기    헬스장        5      20
  오프라인  팝업스토어    백화점       12      -5
  오프라인  팝업스토어    대학교        6      10



In [4]:
# ✏️ 과제 1-2: 마이너스 성장 항목 추출 + 비율 계산
# 마이너스 성장 항목 필터
negative_df = df[df['Growth'] < 0]

# 매출 합 계산
negative_revenue = negative_df['Revenue'].sum()
total_revenue = df['Revenue'].sum()

# 비율 계산
ratio = (negative_revenue / total_revenue) * 100

print(f"마이너스 성장 항목 매출 비율: {ratio:.2f}%")

마이너스 성장 항목 매출 비율: 70.81%


### ✏️ 과제 1-3: MECE 검증 답안 (이 셀에 직접 작성)

- ME(상호 배타) 충족 여부: 온라인/오프라인, 구독형/단건주문, 자판기/팝업스토어 등 각 Level별 항목이 겹치지 않게 구분되어 있기 때문에 상호 배타성은 충족된다고 볼 수 있다.

- CE(전체 포괄) 충족 여부: 주요 매출 경로는 포함되어 있지만, 기타 유통 채널이나 신규 판매 방식이 포함되지 않을 수도 있어서 완전한 포괄성은 조금 부족할 수 있다.

- 보완이 필요한 점: 이후에 신규 채널이나 기타 매출 항목을 추가해 전체 매출 구조를 포괄적으로 반영하는 것이 필요하다고 생각한다.

---

## 과제 2. Why Tree: 이탈 원인 분석 (8점)

### 요구사항

1. **(3점)** 이탈 사유를 **Reason_Category별로 합계와 비율(%)**을 계산하여 출력하세요.

2. **(5점)** 아래 형식의 **Why Tree**를 작성하고, **수평 막대 차트**로 시각화하세요.

```
왜 고객이 이탈하는가? (총 ??건)
├─ 서비스품질 (??건, ??%)
│   ├─ 배송 지연 (??건)
│   └─ 포장 파손 (??건)
├─ 제품 (??건, ??%)
│   └─ ...
...
```

In [5]:
# ✏️ 과제 2-1: 카테고리별 합계 + 비율 계산
# 데이터 준비
df_reason = churn.copy()
df_reason.rename(columns={'Churn_Count': 'Count'}, inplace=True)

# 카테고리별 합계
category_sum = df_reason.groupby('Reason_Category')['Count'].sum()

# 전체 건수
total = df_reason['Count'].sum()

# 비율 계산
category_ratio = (category_sum / total) * 100

# 출력
for cat in category_sum.index:
    print(f"{cat}: {category_sum[cat]}건 ({category_ratio[cat]:.2f}%)")

가격: 19건 (13.77%)
경쟁사: 12건 (8.70%)
서비스품질: 77건 (55.80%)
제품: 30건 (21.74%)


In [6]:
# ✏️ 과제 2-2: Why Tree 텍스트 출력 + 수평 막대 차트 시각화
# Reason_Category별 상세 집계
print(f"왜 고객이 이탈하는가? (총 {total}건)")

for cat in category_sum.index:
    print(f"├─ {cat} ({category_sum[cat]}건, {category_ratio[cat]:.2f}%)")

    # 해당 카테고리의 모든 항목 추출
    sub = df_reason[df_reason['Reason_Category'] == cat]
    # Reason_Detail별 합계
    sub_sum = sub.groupby('Reason_Detail')['Count'].sum()

    for detail in sub_sum.index:
        print(f"│   ├─ {detail} ({sub_sum[detail]}건)")

# 수평 막대 차트 시각화
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=category_sum.values,
    y=category_sum.index,
    orientation='h',
    marker=dict(
        color='white',
        line=dict(color='black', width=1)
    )
))

fig.update_layout(
    title='이탈 사유 카테고리별 분포',
    xaxis_title='건수',
    yaxis_title='카테고리',
    font=dict(color='black')
)

fig.show()

왜 고객이 이탈하는가? (총 138건)
├─ 가격 (19건, 13.77%)
│   ├─ 가격 대비 양 부족 (14건)
│   ├─ 가격 인상 불만 (5건)
├─ 경쟁사 (12건, 8.70%)
│   ├─ 경쟁사 전환 (9건)
│   ├─ 대형마트 유사 상품 (3건)
├─ 서비스품질 (77건, 55.80%)
│   ├─ 배송 지연 (47건)
│   ├─ 포장 파손 (30건)
├─ 제품 (30건, 21.74%)
│   ├─ 맛 불만족 (8건)
│   ├─ 메뉴 반복 (22건)


### ✏️ 과제 2 해석 (이 셀에 직접 타이핑)

- 이탈 원인 1위 카테고리와 그 비율은? 
 : 서비스품질이 77건이며 전체의 55.8%으로 가장 높은 비율을 보여주고 있다.

- 이 카테고리 안에서 가장 건수가 많은 세부 사유는 무엇이며, 이를 줄이기 위해 가장 먼저 해야 할 일은? (1~2문장)
: 서비스품질에서 배송 지연이 47건으로 가장 건수가 많으며, 이를 줄이기 위해 물류 시스템 개선 및 배송 안정성 확보가 가장 먼저 필요하다고 생각한다.

---

## 과제 3. How Tree: 해결책 도출 (8점)

### 요구사항

1. **(5점)** 과제 2에서 가장 큰 이탈 원인 **1개**에 대해 **How Tree**를 마크다운으로 작성하세요.
   - 최소 2단계 깊이 (시간축: 단기/장기)

```
어떻게 [원인]을 해결할 것인가?
├─ 단기 대응 (1개월 내)
│   ├─ 행동 1
│   └─ 행동 2
└─ 장기 개선 (3개월)
    ├─ 행동 3
    └─ 행동 4
```

2. **(3점)** 가장 효과가 클 해결책 **1개**를 선정하고, 그 근거를 데이터(이탈 건수 등)를 인용하여 2문장으로 서술하세요.

### ✏️ 과제 3-1: How Tree (이 셀에 직접 작성)

어떻게 배송 지연을 해결할 것인가?
├─ 단기 대응 (1개월 내)
│   ├─ 배송 지연 발생 시 쿠폰 등 보상 제공으로 고객 불만 완화
│   └─ 배송 인력과 물량을 일시적으로 늘려 지연을 줄이기
└─ 장기 개선 (3개월)
    ├─ 물류 시스템 개선 및 배송 경로 최적화 적용
    └─ 자체 물류 체계를 구축하여 전반적인 배송 안정성 확보

### ✏️ 과제 3-2: 최우선 해결책 + 근거

최우선 해결책은 물류 시스템을 개선해 배송 안정성을 확보하는 것이라고 생각한다. 배송 지연은 47건으로 전체 이탈 중 가장 크게 차지하고 있어, 해결하는 게 이탈 감소에 가장 큰 영향을 줄 수 있다.

---

## 채점 기준

| 과제 | 배점 | 채점 포인트 |
|------|------|------------|
| 과제 1. MECE + 매출 분해 | 9점 | MECE 판단(3) + 시각화(3) + 마이너스 분석(3) |
| 과제 2. Why Tree | 8점 | 집계(3) + 트리+시각화(5) |
| 과제 3. How Tree | 8점 | 해결책 구조(5) + 최우선 행동(3) |
| **합계** | **25점** | |

---

## 참고: 예상 정답값

**과제 1:**
- 전체 매출: 161억 원
- 마이너스 성장 4개 항목 합계: 114억 원 → 전체의 **약 71%**
- 정기배송(45억, -12%)이 가장 큰 하락 항목

**과제 2 이탈 카테고리(3개월 합계):**

| 카테고리 | 건수 | 비율 |
|---------|------|------|
| 서비스품질 | 77건 | 56% |
| 제품 | 30건 | 22% |
| 가격 | 19건 | 14% |
| 경쟁사 | 12건 | 9% |

- 서비스품질이 56%로 1위 → 그 중 배송 지연(47건)이 최다